# 리포트 35 — 시간표본마다 자세를 새로 놓고 다시 쏘아 슬로타임 복소열을 만든다

> ### 한 일
> **로터 위상을 시간표본마다 다시 놓고 광선을 다시 쏘아 되돌아오는 복소 신호의 느린 시간축을 만들었다.**

### 결과
1. 헤드라인 칸은 DJI Matrice 4E [^1] 를 배 쪽에서 본 것이다 — 방위 0 도 [^2] · 앙각 -15 도 [^3].
2. 호버 3800 rpm [^4] 에서 운동학이 예측하는 날개끝 주파수는 1230 Hz [^5], 블레이드 통과율은 126.7 Hz [^6] 다.
3. 표본율 5000 Hz [^7] 로 2526 개 [^8] 를 이어 붙여 창 길이 0.505 s [^9] 를 얻었다.
4. 그 창이 주는 도플러 분해능은 1.98 Hz [^10] 이고, 날개끝까지 621 칸 [^11] 이 든다.
5. 조립을 싸게 만든 것은 `src/articulated_fast.py` 다 — 드론을 한 번 짓고 위상마다 행렬곱만 한다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 슬로타임 복소열 | 시간표본마다 로터 위상을 다시 놓고 광선을 다시 쏜다 — 위상 하나짜리 표를 쓰지 않으므로 로터마다 회전수를 다르게 줄 수 있다 |
| 무엇이 그것을 가능하게 했나 | `src/articulated_fast.py` — 드론을 한 번 짓고 위상마다 행렬곱만 한다. 정점 배열이 옛 함수와 비트 단위로 같다 |
| 도플러 분해능 | 창에 든 블레이드 주기 수가 정한다. 표본 수를 늘려도 안 좋아진다 |
| 헤드라인 기체 선택 | DJI Matrice 4E — 프롭·벨 겹침이 0.01 % 로 정리됐고 1차 실측 표적이다 |

### 재현

```bash
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/report15b_microdoppler_recompute.py
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/report15b_stamp_provenance.py
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/build_report15b_figs.py
```

| | |
|---|---|
| 출력 | `outputs/report15b_microdoppler.json`, `outputs/report15b_series.npz` |
| 소요 | 약 25 분 (GPU 1장 — 광선 추적이 6칸 × 4팔) |
| 비고 | 산출물이 자기가 어떤 메쉬로 계산됐는지 지문을 함께 적는다(`mesh_provenance`) — 계산 도중 메쉬가 바뀌면 스스로 경고한다 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| 앞 편 | [편 34 «스톡 도플러»](34_md-paths-doppler.ipynb) — 왜 이 길로 가나 |

---

## 무엇을 재는가

표적이 제자리에 떠 있어도 날개는 돈다. 날개 표면의 점들이 시간에 따라 자리를 바꾸므로 왕복 위상이 변조되고, 그것이 되돌아오는 신호의 느린 시간축에 실린다. 우리는 그 열을 **시간표본마다 자세를 새로 놓고 광선을 다시 쏘아** 만든다.

왜 그렇게까지 하는가. 로터마다 회전수를 다르게 주려면 드론 전체 자세가 각도 하나의 함수가 아니게 되고, 그러면 «위상 하나짜리 표를 미리 만들어 두고 조회한다» 는 지름길이 막힌다. 조립을 싸게 만들어 그 지름길을 버렸다.

## 헤드라인 칸의 운동학

| 무엇을 | 값 |
|---|---|
| 기체 · 자세 | DJI Matrice 4E [^1] · 배 쪽 |
| 방위 · 앙각 | 0° [^2] · -15° [^3] |
| 호버 회전수 | 3800 rpm [^4] |
| 날개끝 속도 | 54.52 m/s [^12] |
| 날개끝 주파수 | 1230 Hz [^5] |
| 블레이드 통과율 | 126.7 Hz [^6] |

## 창이 분해능을 정한다

| 무엇을 | 값 |
|---|---|
| 표본율 | 5000 Hz [^7] |
| 표본 수 | 2526 개 [^8] |
| 창 길이 | 0.505 s [^9] |
| 도플러 분해능 | 1.98 Hz [^10] |
| 날개끝까지 든 칸 수 | 621 칸 [^11] |

분해능은 «창에 든 블레이드 주기 수» 가 정한다. 같은 창 안에서 표본을 촘촘히 해도 칸이 좁아지지 않으므로, 능선을 가르려면 창을 늘린다.

## 전처리를 어떻게 했는가

마이크로도플러 그림은 전처리가 답을 바꾼다. 그래서 규약을 적어 둔다.

| 단계 | 우리가 한 것 | 왜 |
|---|---|---|
| 채널 | 전체 드론과 프로펠러만을 따로 | 동체가 블레이드를 덮는다 |
| 0 도플러 | **살린다** | 동체 선이 읽기의 기준이다 |
| 조각 길이 | 블레이드 13 주기 | 능선 사이에 13 빈이 들어 빗살이 안 뭉갠다 |
| 창·제로패딩 | Hann · 4배 | 누설을 줄이고 주파수축을 매끈하게 |
| 색역 | 60 dB | 동체 선을 0 dB 로 두고 능선을 그 아래에서 읽는다 |
| 정규화 | 한 그림 안에서 공통 | 두 패널을 나란히 놓고 비교할 수 있게 |

## 검출 축의 전처리는 따로 있다

정적 성분을 지우는 슬로타임 고역통과(MTI)는 `src/microdoppler_proc.py` 에 따로 있다 — **검출 축**에서 쓴다. ⚠ 그 노치는 호버하는 표적의 동체도 함께 지우므로 탐지에서는 대가가 된다.

⚠ 선행 구현의 **처리 파라미터**는 그 시스템의 자원격자에 맞춰진 값이라 그대로 옮기지 않았다. 우리가 가져온 것은 그림을 읽는 순서이고, 차단주파수 같은 것은 우리 물리에서 정했다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 같은 절차를 두 엔진에 태워 무늬를 맞댄다 | 위상을 광선 엔진에 맡기고 세기를 PO 커널에 맡기는 분업이 근거를 얻는다 | [편 36 «두 엔진»](36_md-two-engines.ipynb) |
| 로터마다 회전수를 다르게 준다 | 무늬가 시간에 따라 변하는 데 무엇이 필요한지가 갈린다 | [편 37 «회전수 축»](37_md-rpm.ipynb) |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 12개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.name` | DJI Matrice 4E |
| [^2] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.az_deg` | 0 |
| [^3] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.el_deg` | -15 |
| [^4] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.rpm` | 3800 |
| [^5] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.f_tip` | 1230 |
| [^6] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.f_flash` | 126.7 |
| [^7] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.prf` | 5000 |
| [^8] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.n_t` | 2526 |
| [^9] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.duration_s` | 0.5053 |
| [^10] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.doppler_resolution_hz` | 1.979 |
| [^11] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.bins_to_ftip` | 621.3 |
| [^12] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.v_tip` | 54.52 |